# Discount curves (OIS / risk-free discounting)

Deep-dive reference: **discount factor curves** for present value and curve-implied rates.

## Concept

A **discount curve** maps time (in year fractions from a base date) to **discount factors** \(DF(t)\): the present value of one unit of currency paid at \(t\). From \(DF\) you recover **zero rates** and **implied forward rates** between two times. OIS curves are the usual risk-free discounting choice for many instruments in a single currency.

## API walkthrough

`DiscountCurve` is built from `(time_years, discount_factor)` knots. Use `df(t)` for the discount factor, `zero(t)` for the continuously compounded zero rate, and `forward_rate(t1, t2)` for the forward between two times.

In [ ]:
from datetime import date

from finstack.core.market_data import DiscountCurve

curve = DiscountCurve(
    "USD-OIS",
    date(2024, 1, 1),
    [(0.0, 1.0), (0.5, 0.975), (1.0, 0.95), (2.0, 0.90), (5.0, 0.75), (10.0, 0.50)],
    day_count="act_365f",
)
print("curve:", curve)
print("df(1.0) =", curve.df(1.0))
print("zero(1.0) =", curve.zero(1.0))

In [ ]:
tenors = [0.25, 0.5, 1.0, 2.0, 5.0, 10.0]
print("Tenor (y)\tDF(t)\t\tzero(t)")
for t in tenors:
    print(f"{t}\t\t{curve.df(t):.6f}\t{curve.zero(t):.6f}")

t1, t2 = 1.0, 5.0
fwd = curve.forward_rate(t1, t2)
print()
print(f"Continuous forward {t1}y -> {t2}y: {fwd:.6f}")

usd = curve
eur = DiscountCurve(
    "EUR-ESTR",
    date(2024, 1, 1),
    [(0.0, 1.0), (0.5, 0.978), (1.0, 0.955), (2.0, 0.91), (5.0, 0.78), (10.0, 0.55)],
    day_count="act_365f",
)
print()
print("Side-by-side at 5Y: USD-OIS df =", usd.df(5.0), "| EUR-ESTR df =", eur.df(5.0))
print("Side-by-side at 5Y: USD zero =", usd.zero(5.0), "| EUR zero =", eur.zero(5.0))

Beyond the last knot (\(t=10\)y here), discount factors and zeros follow the curve’s **extrapolation** rule (often **flat forward** by default). Querying **\(t=15\)** and **\(t=20\)** years shows that behavior explicitly.

In [ ]:
for t in (15.0, 20.0):
    print(f"t={t}y  df(t)={curve.df(t):.8f}  zero(t)={curve.zero(t):.6f}")

## Practical example

Discount a single future cashflow: \(PV = C \cdot DF(T)\).

In [ ]:
cashflow = 1_000_000.0
maturity_years = 3.0
df_t = curve.df(maturity_years)
pv = cashflow * df_t
print(f"Cashflow USD {cashflow:,.0f} paid in {maturity_years}y")
print(f"DF({maturity_years}) = {df_t:.6f}")
print(f"Present value = USD {pv:,.2f}")

## Takeaways

- **Knots** are \((t, DF)\); interpolation and extrapolation follow the curve builder defaults (`monotone_convex` / `flat_forward` unless you override).
- **`df` / `zero` / `forward_rate`** are the three core queries for PV, par-rate intuition, and hedging/funding views.
- **Multi-currency** workflows use one discount curve per currency (e.g. USD-OIS vs EUR-ESTR); cross-currency needs FX and often a basis, which is outside this single-curve reference.

## Calibration from market quotes

In practice, discount curves are **bootstrapped** from observable market instruments (deposit rates and swap par rates) rather than constructed from hand-picked knots.  The calibration engine accepts a JSON plan specifying quote data and curve parameters, then solves for the discount factors that reprice each instrument exactly.

The quote format uses:
- `"class": "rates"` with `"type": "deposit"` or `"type": "swap"`
- `"pillar"` as `{"tenor": {"count": N, "unit": "months"|"years"}}` or `{"date": "YYYY-MM-DD"}`

In [ ]:
import json

from finstack.valuations import calibrate

def tenor(count, unit):
    """Helper to build a pillar tenor dict."""
    return {"tenor": {"count": count, "unit": unit}}

plan = {
    "schema": "finstack.calibration/2",
    "plan": {
        "id": "usd-ois-bootstrap",
        "description": "USD OIS discount curve from deposits and swaps",
        "quote_sets": {
            "ois": [
                {"class": "rates", "type": "deposit", "id": "DEP-1M", "index": "USD-SOFR", "pillar": tenor(1, "months"), "rate": 0.0525},
                {"class": "rates", "type": "deposit", "id": "DEP-3M", "index": "USD-SOFR", "pillar": tenor(3, "months"), "rate": 0.0530},
                {"class": "rates", "type": "deposit", "id": "DEP-6M", "index": "USD-SOFR", "pillar": tenor(6, "months"), "rate": 0.0535},
                {"class": "rates", "type": "swap", "id": "SWP-1Y", "index": "USD-SOFR-OIS", "pillar": tenor(1, "years"), "rate": 0.0540},
                {"class": "rates", "type": "swap", "id": "SWP-2Y", "index": "USD-SOFR-OIS", "pillar": tenor(2, "years"), "rate": 0.0480},
                {"class": "rates", "type": "swap", "id": "SWP-5Y", "index": "USD-SOFR-OIS", "pillar": tenor(5, "years"), "rate": 0.0420},
                {"class": "rates", "type": "swap", "id": "SWP-10Y", "index": "USD-SOFR-OIS", "pillar": tenor(10, "years"), "rate": 0.0410},
            ]
        },
        "steps": [{
            "id": "USD-OIS",
            "quote_set": "ois",
            "kind": "discount",
            "curve_id": "USD-OIS",
            "currency": "USD",
            "base_date": "2025-01-15",
            "method": "Bootstrap",
            "interpolation": "linear",
            "extrapolation": "flat_forward",
        }],
        "settings": {"use_parallel": False},
    },
}

result = calibrate(json.dumps(plan))
print("Success:", result.success)
print("Iterations:", result.iterations)
print("Max residual:", f"{result.max_residual:.2e}")
print()

cal_curve = result.market.get_discount("USD-OIS")
print("Calibrated curve term structure:")
for t in [0.25, 0.5, 1.0, 2.0, 5.0, 10.0]:
    print(f"  t={t:5.2f}y  DF={cal_curve.df(t):.8f}  zero={cal_curve.zero(t):.6f}")

In [ ]:
# Inspect the per-step calibration report
print(result.report_to_dataframe().to_string(index=False))